[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sergiovillanueva/Modelos_Fundacionales/blob/main/3_Multimodal.ipynb)

**Día 1 · Cuaderno 3 de 3** · Acompaña al tema 3 de la web: [Modelos multimodales](https://datamecum.sergiovillanuevalopez.com/temas/03-multimodalidad/)

# Modelos multimodales: visión y lenguaje

Los modelos multimodales entienden a la vez imágenes y texto. Eso permite pedirles cosas en lenguaje natural, sin una lista cerrada de clases y sin entrenar nada. Vemos cuatro, de menos a más capaz:

| Modelo | Qué hace |
|---|---|
| **CLIP** | Dice cuál de varios textos describe mejor una imagen (clasificación *zero-shot*) |
| **BLIP** | Describe una imagen con una frase |
| **Grounding DINO** | Detecta los objetos que le nombras en un texto |
| **Qwen3.5** | Un modelo de visión y lenguaje (VLM): hace lo de los tres anteriores y además lee texto, razona, compara imágenes y responde en JSON |

CLIP, BLIP y Grounding DINO esperan el texto **en inglés**, que es el idioma en el que se entrenaron. Qwen entiende también el castellano.

**Antes de empezar**

1. Activa la GPU: menú *Entorno de ejecución* → *Cambiar tipo de entorno de ejecución* → *GPU T4* → *Guardar*.
2. Ejecuta las celdas en orden, de arriba abajo, con `Mayúsculas + Intro`.
3. No hay que instalar nada: todas las librerías que usamos ya vienen en Colab. Los modelos se descargan solos desde Hugging Face la primera vez que se usan.

Este cuaderno es tuyo: al abrirlo desde el enlace se crea una copia que solo ves tú, y puedes cambiar lo que quieras sin estropear el original.

## Preparación

In [ ]:
import json
import torch
import requests
import transformers
import matplotlib.pyplot as plt
from io import BytesIO
from pathlib import Path
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
# En GPU usamos media precisión (float16): ocupa la mitad de memoria y va más rápido.
dtype = torch.float16 if device == "cuda" else torch.float32
print("Usando:", device)
transformers.logging.set_verbosity_error()   # oculta avisos internos que no afectan al resultado

ASSETS = "https://github.com/sergiovillanueva/Modelos_Fundacionales/raw/main/assets/"


def cargar_imagen(nombre_o_url):
    """Abre una imagen del curso en RGB.

    Si el cuaderno se ejecuta dentro de la carpeta del curso descargada (por ejemplo, sin internet),
    la toma de assets/; si no, la descarga del repositorio del curso o de la URL indicada.
    """
    local = Path("assets") / Path(nombre_o_url).name
    if local.exists():
        return Image.open(local).convert("RGB")
    url = nombre_o_url if nombre_o_url.startswith("http") else ASSETS + nombre_o_url
    respuesta = requests.get(url, timeout=30)
    respuesta.raise_for_status()
    return Image.open(BytesIO(respuesta.content)).convert("RGB")


def dibujar_cajas(imagen, cajas, etiquetas, titulo=""):
    """Dibuja cajas [x1, y1, x2, y2] sobre la imagen, con su etiqueta encima. Un color por clase."""
    colores = {}
    plt.figure(figsize=(12, 8))
    plt.imshow(imagen)
    eje = plt.gca()
    for (x1, y1, x2, y2), etiqueta in zip(cajas, etiquetas):
        clase = etiqueta.split()[0]
        color = colores.setdefault(clase, plt.cm.tab10(len(colores) % 10))
        eje.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color=color, linewidth=3))
        eje.text(x1, y1, etiqueta, color="white", fontsize=10, va="bottom",
                 bbox=dict(facecolor=color, edgecolor="none", pad=1.5))
    plt.title(titulo)
    plt.axis("off")
    plt.show()


image_cars = cargar_imagen("cars.jpg")
image_person_cars = cargar_imagen("person_cars.jpg")
image_person_dog = cargar_imagen("person_dog.jpg")
image_dog = cargar_imagen("dog.jpg")
image_fruits = cargar_imagen("fruits.jpg")
image_board = cargar_imagen("board.jpg")
image_carpet_nok = cargar_imagen("carpet_nok.jpg")
print("Imágenes cargadas")

## CLIP: clasificar sin entrenar

CLIP se entrenó con 400 millones de pares de imagen y texto sacados de internet. Aprendió a poner en el mismo espacio la imagen de un perro y la frase «a photo of a dog», de modo que se puede medir cuánto se parecen.

Para clasificar le damos una imagen y varias descripciones, y nos dice cuál encaja mejor. Las clases las inventas tú en cada llamada: por eso se llama *zero-shot*, sin ejemplos de entrenamiento.

Paper: [Radford et al., 2021](https://arxiv.org/abs/2103.00020)

In [ ]:
from transformers import CLIPModel, CLIPProcessor

clip_name = "openai/clip-vit-base-patch32"
clip_processor = CLIPProcessor.from_pretrained(clip_name)
clip_model = CLIPModel.from_pretrained(clip_name).to(device)
print("CLIP cargado")

In [ ]:
def clasificar_con_clip(imagen, textos):
    """Devuelve la probabilidad de cada texto para la imagen y la muestra con un gráfico de barras."""
    entradas = clip_processor(text=textos, images=imagen, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        probabilidades = clip_model(**entradas).logits_per_image.softmax(dim=1)[0].cpu()

    figura, ejes = plt.subplots(1, 2, figsize=(14, 4), gridspec_kw={"width_ratios": [1, 1.2]})
    ejes[0].imshow(imagen)
    ejes[0].axis("off")
    ejes[1].barh(textos, probabilidades, color="#2727c7")
    ejes[1].invert_yaxis()
    ejes[1].set_xlim(0, 1)
    ejes[1].set_xlabel("probabilidad")
    plt.tight_layout()
    plt.show()
    return dict(zip(textos, probabilidades.tolist()))


_ = clasificar_con_clip(image_cars, ["a parking lot with cars", "a factory", "an office", "a park"])

Las clases son las que quieras. Prueba a cambiar los textos:

In [ ]:
_ = clasificar_con_clip(image_fruits, ["a photo of fruit", "a photo of cars", "a photo of a dog"])
# _ = clasificar_con_clip(image_carpet_nok, ["a clean carpet", "a carpet with a defect"])
# _ = clasificar_con_clip(image_board, ["an electronic circuit board", "a kitchen", "a car engine"])

### Buscar fotos escribiendo

Con la misma idea se puede buscar en una colección de fotos: en vez de comparar una imagen con varios textos, se compara un texto con muchas imágenes y se ordenan de más a menos parecida. Así funcionan los buscadores de fotos que entienden «perro en la playa» sin que nadie haya etiquetado nada.

Probamos con las imágenes del curso.

In [ ]:
nombres = ["cars.jpg", "person_cars.jpg", "person_dog.jpg", "dog.jpg", "dog2.jpg", "fruits.jpg", "fruits3.jpg",
           "bananas.jpg", "board.jpg", "carpet_ok.jpg", "park.jpg", "person_banana.jpg", "cut.jpg"]
coleccion = [cargar_imagen(nombre) for nombre in nombres]


def buscar(texto, cuantas=3):
    """Ordena la colección según lo bien que encaja cada foto con el texto y enseña las mejores."""
    entradas = clip_processor(text=[texto], images=coleccion, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        parecido = clip_model(**entradas).logits_per_text[0].softmax(dim=0).cpu()
    mejores = parecido.topk(cuantas)

    figura, ejes = plt.subplots(1, cuantas, figsize=(5 * cuantas, 4))
    for eje, valor, indice in zip(ejes, mejores.values, mejores.indices):
        eje.imshow(coleccion[indice])
        eje.set_title(f"{nombres[indice]} ({valor:.0%})")
        eje.axis("off")
    plt.suptitle(f"«{texto}»")
    plt.tight_layout()
    plt.show()


buscar("a dog sleeping on a sofa")
buscar("an aerial view of a city park")
# buscar("an electronic circuit")
# buscar("a woman holding a banana")

## BLIP: describir una imagen

BLIP genera una frase que describe la imagen. Sirve para crear descripciones automáticas, indexar un archivo de fotos o hacer accesibles las imágenes para personas ciegas.

Paper: [Li et al., 2022](https://arxiv.org/abs/2201.12086)

In [ ]:
from transformers import BlipForConditionalGeneration, BlipProcessor

blip_name = "Salesforce/blip-image-captioning-base"
blip_processor = BlipProcessor.from_pretrained(blip_name)
blip_model = BlipForConditionalGeneration.from_pretrained(blip_name, dtype=dtype).to(device)
print("BLIP cargado")

In [ ]:
def describir(imagen):
    """Genera una frase que describe la imagen."""
    entradas = blip_processor(images=imagen, return_tensors="pt").to(device, dtype)
    with torch.no_grad():
        salida = blip_model.generate(**entradas, max_new_tokens=40)
    frase = blip_processor.decode(salida[0], skip_special_tokens=True)

    plt.imshow(imagen)
    plt.title(frase)
    plt.axis("off")
    plt.show()
    return frase


_ = describir(image_person_cars)
_ = describir(image_board)

## Grounding DINO: detectar lo que nombras

Grounding DINO es un detector como el del cuaderno 1, pero en vez de 80 clases fijas detecta lo que le escribas: «kiwi», «yellow car», «person with a red shirt».

El texto tiene un formato concreto: **en minúsculas y cada objeto terminado en punto**, por ejemplo `"kiwi. apple."`.

Ojo con el nombre: este DINO es un detector y no tiene nada que ver con el DINO autosupervisado del cuaderno 4.

Paper: [Liu et al., 2023](https://arxiv.org/abs/2303.05499)

In [ ]:
from transformers import AutoModelForZeroShotObjectDetection, AutoProcessor

dino_name = "IDEA-Research/grounding-dino-base"
dino_processor = AutoProcessor.from_pretrained(dino_name)
dino_model = AutoModelForZeroShotObjectDetection.from_pretrained(dino_name).to(device)
print("Grounding DINO cargado")

In [ ]:
def detectar_por_texto(imagen, texto, umbral=0.35):
    """Detecta los objetos nombrados en el texto, por ejemplo "kiwi. apple."."""
    entradas = dino_processor(images=imagen, text=texto, return_tensors="pt").to(device)
    with torch.no_grad():
        salidas = dino_model(**entradas)
    resultado = dino_processor.post_process_grounded_object_detection(
        salidas, entradas.input_ids, threshold=umbral, text_threshold=0.25, target_sizes=[imagen.size[::-1]]
    )[0]

    cajas = resultado["boxes"].tolist()
    etiquetas = [f"{nombre} {puntuacion:.2f}" for nombre, puntuacion in zip(resultado["labels"], resultado["scores"])]
    dibujar_cajas(imagen, cajas, etiquetas, f"«{texto}»: {len(cajas)} objetos")
    return resultado


_ = detectar_por_texto(image_fruits, "kiwi. apple.")
_ = detectar_por_texto(image_person_cars, "yellow car. woman.")

## Qwen3.5: un modelo para casi todo

Hasta aquí cada modelo hacía una cosa: CLIP clasifica, BLIP describe y Grounding DINO detecta. Un VLM (*Vision-Language Model*) es un modelo de lenguaje, como los de los chats, al que se le ha añadido la vista. Con el mismo modelo, y cambiando solo el texto que le escribes, puede:

- describir una escena y responder preguntas sobre ella,
- razonar sobre lo que ve,
- leer el texto que aparece en la imagen,
- comparar dos imágenes,
- devolver la respuesta en JSON, listo para un programa,
- localizar objetos descritos con palabras.

Usamos **Qwen3.5-2B**, de Alibaba (febrero de 2026, licencia Apache 2.0). Es de los VLM más pequeños que funcionan bien: son 4,5 GB de descarga, así que **la primera carga tarda unos minutos**. Después, cada respuesta tarda entre uno y cinco segundos en la T4.

Si quieres probar otro, cambia `vlm_name` por `"Qwen/Qwen3-VL-2B-Instruct"`; el resto del código sirve igual.

In [ ]:
from transformers import AutoModelForImageTextToText

vlm_name = "Qwen/Qwen3.5-2B"
vlm_processor = AutoProcessor.from_pretrained(vlm_name)
vlm_model = AutoModelForImageTextToText.from_pretrained(vlm_name, dtype=dtype, device_map="auto")
print(f"{vlm_name} cargado")

La función `preguntar` envía una o varias imágenes y un texto, y devuelve lo que contesta el modelo.

**Cómo escribir el prompt para que acierte:** una sola tarea por pregunta, concreta, y diciendo cómo quieres la respuesta («en una frase», «solo las etiquetas», «en JSON»). Si no le pones límite, un VLM tiende a extenderse.

In [ ]:
def preguntar(imagenes, pregunta, max_new_tokens=200):
    """Envía una imagen (o una lista de imágenes) y una pregunta al VLM y devuelve su respuesta."""
    if not isinstance(imagenes, list):
        imagenes = [imagenes]
    contenido = [{"type": "image", "image": imagen} for imagen in imagenes]
    contenido.append({"type": "text", "text": pregunta})
    entradas = vlm_processor.apply_chat_template(
        [{"role": "user", "content": contenido}],
        add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt",
    ).to(vlm_model.device)
    with torch.no_grad():
        generado = vlm_model.generate(**entradas, max_new_tokens=max_new_tokens, do_sample=False)
    # La salida incluye la pregunta; nos quedamos solo con lo que ha escrito el modelo
    respuesta = generado[0][entradas["input_ids"].shape[1]:]
    return vlm_processor.decode(respuesta, skip_special_tokens=True).strip()


def mostrar(imagenes):
    """Enseña una o varias imágenes en fila."""
    if not isinstance(imagenes, list):
        imagenes = [imagenes]
    figura, ejes = plt.subplots(1, len(imagenes), figsize=(6 * len(imagenes), 4), squeeze=False)
    for eje, imagen in zip(ejes[0], imagenes):
        eje.imshow(imagen)
        eje.axis("off")
    plt.show()

### Describir y razonar

La misma escena del semáforo de antes. Primero una descripción, después una pregunta que obliga a razonar: para contestarla tiene que ver que hay una persona en el paso de cebra y entender qué implica.

In [ ]:
mostrar(image_person_cars)
for pregunta in [
    "Describe the scene in two sentences.",
    "Is it safe for the cars to drive now? Answer in one sentence and explain why.",
]:
    print("P:", pregunta)
    print("R:", preguntar(image_person_cars, pregunta), "\n")

### Leer texto

Un VLM lee el texto de la imagen y entiende para qué sirve. Si además le pides JSON con los campos que quieres, la respuesta se puede guardar directamente en una base de datos: así se automatiza la lectura de etiquetas, albaranes o placas de características.

In [ ]:
mostrar(image_board)
print(preguntar(image_board, "What does the text on this board tell the user to do, and what do you think the board measures? Answer in one sentence."))
print(preguntar(image_board, 'Return only JSON with the text printed on this board: {"product": "...", "website": "..."}'))

### Lo que sabe del mundo

Aprendió con millones de imágenes y textos de internet, así que reconoce lugares, marcas u objetos conocidos sin que se lo enseñes.

In [ ]:
image_park = cargar_imagen("park.jpg")
mostrar(image_park)
print(preguntar(image_park, "Which city is this, and what is the large green area? Answer in one sentence."))

### Comparar dos imágenes

Se le pueden pasar varias imágenes en la misma pregunta. Aquí, una tela buena y una con un defecto: le pedimos que diga cuál está dañada y dónde.

In [ ]:
image_carpet_ok = cargar_imagen("carpet_ok.jpg")
mostrar([image_carpet_ok, image_carpet_nok])
print(preguntar([image_carpet_ok, image_carpet_nok],
                "Which of the two images shows damaged fabric, the first or the second? "
                "Where is the damage? Answer in two sentences."))

### Un informe de inspección en JSON

Lo mismo, pero con la salida que necesitaría un programa de control de calidad: un JSON con campos fijos.

In [ ]:
informe = preguntar(image_carpet_nok, 'You inspect fabric in a factory. '
                    'Return only JSON: {"defect": true or false, "type": "...", "location": "..."}')
print(informe)

### También en castellano

A diferencia de CLIP, BLIP y Grounding DINO, Qwen entiende y responde en castellano. Para respuestas cortas funciona bien; si buscas la máxima precisión, en inglés suele acertar algo más.

In [ ]:
print(preguntar(image_person_cars, "¿Pueden arrancar ya los coches? Responde en una frase y explica por qué."))
print(preguntar(image_carpet_nok, "¿Tiene algún defecto esta tela? Si lo tiene, di cuál y dónde, en una frase."))

### Detectar con instrucciones

Si le pedimos las cajas en JSON, el VLM se convierte en un detector al que se le puede pedir casi cualquier cosa: no solo «las frutas», también «los cítricos», «el bolso de la mujer» o «el chip principal».

**Un detalle importante:** Qwen3.5 devuelve las coordenadas en una escala de 0 a 1000, no en píxeles. Hay que convertirlas al tamaño real de la imagen, y si se olvida, todas las cajas aparecen apelotonadas en una esquina.

In [ ]:
# Los "..." importan: si el ejemplo dijera "name", el modelo copiaría esa palabra como etiqueta.
FORMATO = ' Return only JSON in this format: [{"bbox_2d": [x1, y1, x2, y2], "label": "..."}]'


def leer_json(texto):
    """Extrae la lista JSON de la respuesta, venga o no entre ```json ... ```."""
    if "```" in texto:
        texto = texto.split("```")[1].removeprefix("json")
    try:
        datos = json.loads(texto)
    except json.JSONDecodeError:
        return []
    datos = datos if isinstance(datos, list) else [datos]
    return [d for d in datos if isinstance(d, dict) and "bbox_2d" in d]


def detectar_con_vlm(imagen, instruccion):
    """Pide al VLM las cajas en JSON, las pasa de la escala 0-1000 a píxeles y las dibuja."""
    respuesta = preguntar(imagen, instruccion + FORMATO, max_new_tokens=512)
    print("Respuesta del modelo:", respuesta)

    ancho, alto = imagen.size
    cajas, etiquetas = [], []
    for objeto in leer_json(respuesta):
        x1, y1, x2, y2 = objeto["bbox_2d"]
        cajas.append([x1 / 1000 * ancho, y1 / 1000 * alto, x2 / 1000 * ancho, y2 / 1000 * alto])
        etiquetas.append(str(objeto.get("label", "objeto")))

    dibujar_cajas(imagen, cajas, etiquetas, instruccion)
    return cajas, etiquetas


_ = detectar_con_vlm(image_fruits, "Detect every fruit. Use the name of each fruit as its label.")

No hay lista de clases, así que valen descripciones: colores, tipos, partes de un objeto, varias cosas a la vez.

In [ ]:
image_room = cargar_imagen("000000000139.jpg")
_ = detectar_con_vlm(image_room, "Detect the TV, the woman and the vase with the pink flower.")
_ = detectar_con_vlm(image_board, "Detect the main chip and the blue potentiometer.")
_ = detectar_con_vlm(image_carpet_nok, "Detect the defect in the fabric.")
_ = detectar_con_vlm(image_person_cars, "Detect the woman's bag.")
# _ = detectar_con_vlm(image_fruits, "Detect the citrus fruits.")
# _ = detectar_con_vlm(image_person_cars, "Detect the yellow taxi and the white car.")

### Pensar primero, detectar después

Pídele directamente «la fruta que le cayó a Newton en la cabeza» y puede marcar otra: tiene que deducir qué fruta es y localizarla a la vez. Si se lo preguntas por texto, responde «apple» sin dudar.

El truco es repartir el trabajo en dos llamadas: primero la pregunta, después la detección con la respuesta. Sirve con cualquier VLM y para cualquier tarea en la que haya que razonar antes de localizar.

In [ ]:
fruta = preguntar(image_fruits, "Which fruit fell on Newton's head? Answer with one word.").strip(" .").lower()
print("El modelo dice:", fruta)
_ = detectar_con_vlm(image_fruits, f"Detect the {fruta}.")

## Cuándo usar cada uno

| Modelo | Mejor para | A favor | En contra |
|---|---|---|---|
| **CLIP** | Clasificar con clases que eliges tú | Muy rápido y ligero | Solo clasifica la imagen entera |
| **BLIP** | Describir imágenes | Frases naturales | No localiza objetos |
| **Grounding DINO** | Detectar objetos nombrados | Cajas precisas | Entiende nombres, no instrucciones largas |
| **Qwen3.5** | Casi todo: preguntas, lectura de texto, comparar, JSON, detectar con instrucciones | Un solo modelo para muchas tareas, entiende castellano | Más lento y pesado; conviene revisar sus respuestas antes de automatizar |

Regla práctica: usa el modelo más sencillo que resuelva el problema. Si con CLIP basta, no hace falta un VLM.

## Ejercicio

Elige una de las funciones (`clasificar_con_clip`, `describir`, `detectar_por_texto`, `preguntar` o `detectar_con_vlm`) y pruébala con una imagen tuya. Puedes cargarla con `cargar_imagen("https://...")` o subirla:

```python
from google.colab import files
subido = files.upload()
mi_imagen = Image.open(next(iter(subido))).convert("RGB")
```

Ideas: pregunta a Qwen algo que exija razonar («¿qué falta en esta mesa para poder comer?»), compara lo que detecta Grounding DINO con lo que detecta Qwen con la misma frase, o busca una imagen en la que CLIP se equivoque.

In [ ]:
# Tu código aquí

## Resumen

- **CLIP** compara una imagen con textos: clasifica con las clases que inventes, sin entrenar, y busca fotos en una colección escribiendo lo que quieres ver.
- **BLIP** describe una imagen con una frase.
- **Grounding DINO** detecta lo que nombras, con el formato `"objeto. otro objeto."`.
- **Qwen3.5** hace todo lo anterior con un solo modelo y además lee texto, razona, compara imágenes y responde en JSON; sus cajas van de 0 a 1000 y hay que pasarlas a píxeles.
- Todos se cargan desde Hugging Face con `from_pretrained`. Los tres primeros esperan el texto en inglés; Qwen entiende también castellano.

**Fin del día 1.** Mañana empezamos con el cuaderno 4, DINO: [abrir en Colab](https://colab.research.google.com/github/sergiovillanueva/Modelos_Fundacionales/blob/main/4_DINO.ipynb).